# TUS-GAN Section 6.8: Soft Penalty Weight ($\lambda_{\text{rule}}$) Sensitivity Sweep

This notebook trains and evaluates the sensitivity sweep required for **Section 6.8 & Figure 10** of the manuscript:
> **Target Journal**: Elsevier *Transportation Research Part C: Emerging Technologies*

### Experimental Matrix
- **Backbone**: Cell A (TUS-GAN v4 CNN + AC-GAN + 4 Time-Slice Transitions)
- **Sweep Range**: $\lambda_{\text{rule}} \in \{0.1, 1.0, 10.0, 100.0\}$
- **Objective**: Demonstrate that the soft constraint mechanism exhibits a broad stability plateau without knife-edge sensitivity to hyperparameter tuning.

In [ ]:
# ── 1. Environment & Device Setup ──
import os
import sys
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# Enable TF32 for Ampere / Hopper GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# ── 2. Path Setup & Module Imports ──
import os
import sys

# Add v4 backbone and experiment directories to path
for p in ["../../v4", "../v4", "v4"]:
    if os.path.exists(p):
        sys.path.insert(0, os.path.abspath(p))
        break

if os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

from generator import Generator
from critic import Critic, compute_gradient_penalty, critic_loss, generator_loss
from train_sweep import train_sweep_run
from evaluate_sweep import evaluate_checkpoint, save_and_merge_results
from make_figure10_TRC import get_sensitivity_data, render_figure10
print("✅ All modules successfully imported.")


In [ ]:
# ── 3. Run Training Loop for Sensitivity Sweep ──
DATA_PATH = "data/tusgan_encode.npz"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "v4/tusgan_encode.npz"

# Example: Train lambda_rule = 0.1, 1.0, 100.0
# (Uncomment to execute full training on GPU)
# for lam in [0.1, 1.0, 100.0]:
#     cfg = {
#         "data_path": DATA_PATH,
#         "checkpoint_dir": f"checkpoints/lambda_{lam}",
#         "log_dir": f"checkpoints/lambda_{lam}/runs",
#         "lambda_logic": lam,
#         "lambda_rule": lam,
#         "epochs": 250,
#         "batch_size": 512,
#         "seed": 42,
#         "lr": 1e-4,
#         "device": str(device),
#         "num_workers": 2,
#         "noise_dim": 128,
#         "district_embed_dim": 16,
#         "state_embed_dim": 8,
#         "g_base_channels": 256,
#         "d_base_channels": 64,
#         "n_critic": 5,
#         "lambda_gp": 10.0,
#         "lambda_transition": 10.0,
#         "lambda_cond": 1.0,
#         "beta1": 0.0,
#         "beta2": 0.9,
#         "ema_decay": 0.999,
#         "gumbel_temp_start": 1.0,
#         "gumbel_temp_min": 0.1,
#         "gumbel_temp_decay": 0.013,
#         "save_every": 50,
#         "log_every_epoch": 25,
#     }
#     train_sweep_run(cfg)


In [ ]:
# ── 4. Render Elsevier TRC Figure 10 ──
from IPython.display import Image, display

records = get_sensitivity_data("results_sensitivity_sweep.json")
render_figure10(output_dir="outputs", records=records)
display(Image("outputs/figure10_sensitivity_tradeoff.png"))
